# Chapter 6: Design Evals and Deploy

Chapter 5 ended with a question you couldn't answer: how many relevant
papers did v1 miss on Tuesday? The suggestions list couldn't tell you,
because a missed paper doesn't show up anywhere. You'd have to check
every verdict, suggestions *and* skips, against the right answer.

We have the right answers. That's what Chapter 4's labels are for. And
we have every verdict, because Chapter 5 logged all of them. This
chapter puts the two side by side, turns the comparison into a couple
of numbers you can actually reason about, and then uses those numbers
to make a real decision: is v1 ready to ship?

:::{card} Chapter 6 of 12 · Continuous Development

**Where we left off:** v1 runs inside its scope and logs every verdict, but "the suggestions look fine" is an impression, and it can't see missed papers.

**What we fix now:** Compare every verdict against the labels, measure v1 with precision and recall, set a bar it must clear, and ship it at the "suggest" setting.

**By the end, you can:**
- explain precision and recall in plain words, and why accuracy alone is misleading here
- see how a confidence threshold trades one against the other
- decide whether v1 is ready to ship, and say what "shipping" means at this autonomy level
:::

```mermaid
flowchart TB
    subgraph F["Foundations"]
        direction LR
        LOOP[Agent loop] --> MOD[Model decides] --> TOOL[Tools and autonomy]
    end
    subgraph D["Continuous Development"]
        direction LR
        SCP[Scope and data] --> V1[Build v1] --> EVL[Evals and deploy]
    end
    subgraph C["Continuous Calibration, repeats"]
        direction LR
        RUN[Real use] --> ERR[Error patterns] --> FIX[Fixes] --> MET[New metrics]
    end
    F --> D --> C

    classDef active fill:#fef3c7,stroke:#d97706,stroke-width:3px,color:#78350f
    classDef done fill:#dbeafe,stroke:#93c5fd,stroke-width:1px,color:#1e3a8a
    classDef pending fill:#f3f4f6,stroke:#e5e7eb,stroke-width:1px,color:#9ca3af
    class EVL active
    class LOOP,MOD,TOOL,SCP,V1 done
    class RUN,ERR,FIX,MET pending
```

## The fix: check every verdict against its label

An **eval** (short for evaluation) is simple at heart: run the system
on examples where you know the right answer, and count how often it
matches. Everything else is deciding *what* to count.

:::{tip} Try it

The first cell loads the 24 papers with their labels, and v1's verdicts
from Chapter 5. Those verdicts come from Chapter 5's hand-written
stand-in; if you ran a real model in Chapter 5's Colab bonus, you can
paste its confidences into `v1_verdicts` instead and everything below
still works. v1 suggests a paper when its confidence is at least 0.5.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/06-evals-and-deploy.ipynb)

If the code below doesn't let you type into it, use the badge above to
open this same notebook in Google Colab, where you can edit and run it
freely.

The code cells on this page build on each other, so run them in order,
starting from the first one. If a cell stops with a `NameError`, an
earlier cell hasn't been run yet: scroll up, run the cells above it, and
try again.
:::

In [ ]:
papers = [
    {"id": "P01",
     "day": "Mon",
     "title": "RAG Without Chunking",
     "label": True},
    {"id": "P02",
     "day": "Mon",
     "title": "Sparse Mixture-of-Experts at Scale",
     "label": False},
    {"id": "P03",
     "day": "Mon",
     "title": "Retrieval Practice Boosts Exam Scores in Engineering Courses",
     "label": False},
    {"id": "P04",
     "day": "Mon",
     "title": "Grounding Answers in External Documents",
     "label": True},
    {"id": "P05",
     "day": "Mon",
     "title": "Protein Folding with Diffusion Models",
     "label": False},
    {"id": "P06",
     "day": "Mon",
     "title": "A Smaller Vision Transformer",
     "label": False},
    {"id": "P07",
     "day": "Mon",
     "title": "Citation Accuracy in LLM Answers",
     "label": True},
    {"id": "P08",
     "day": "Mon",
     "title": "Faster Training for Speech Models",
     "label": False},
    {"id": "P09",
     "day": "Tue",
     "title": "Better Answers with Less Compute",
     "label": True},
    {"id": "P10",
     "day": "Tue",
     "title": "Neural Architecture Search for Tiny Devices",
     "label": False},
    {"id": "P11",
     "day": "Tue",
     "title": "Dense Retrieval for Product Search",
     "label": False},
    {"id": "P12",
     "day": "Tue",
     "title": "Long-Context Models vs. Retrieval: When Is RAG Still Needed?",
     "label": True},
    {"id": "P13",
     "day": "Tue",
     "title": "Teaching Robots to Fold Laundry",
     "label": False},
    {"id": "P14",
     "day": "Tue",
     "title": "Weather Forecasting with Graph Networks",
     "label": False},
    {"id": "P15",
     "day": "Tue",
     "title": "Hallucination Detection via Self-Consistency",
     "label": False},
    {"id": "P16",
     "day": "Tue",
     "title": "Reranking Retrieved Passages with Small Models",
     "label": True},
    {"id": "P17",
     "day": "Wed",
     "title": "Web-Browsing Agents that Answer from Live Pages",
     "label": True},
    {"id": "P18",
     "day": "Wed",
     "title": "Quantizing LLMs to 3 Bits",
     "label": False},
    {"id": "P19",
     "day": "Wed",
     "title": "Evaluating RAG Pipelines Without Human Labels",
     "label": True},
    {"id": "P20",
     "day": "Wed",
     "title": "Graph Neural Networks for Traffic",
     "label": False},
    {"id": "P21",
     "day": "Wed",
     "title": "Chunk Size Matters: A Study of Document Splitting",
     "label": True},
    {"id": "P22",
     "day": "Wed",
     "title": "Music Generation with Transformers",
     "label": False},
    {"id": "P23",
     "day": "Wed",
     "title": "Code Completion with Repository Context",
     "label": True},
    {"id": "P24",
     "day": "Wed",
     "title": "Speeding Up Diffusion Sampling",
     "label": False},
]

v1_verdicts = {
    "P01": {"confidence": 0.91, "reason": "The paper is about RAG systems and how they index documents."},
    "P02": {"confidence": 0.05, "reason": "The paper is about model architecture and training cost, not documents."},
    "P03": {"confidence": 0.62, "reason": "The paper is about retrieval, which matches the interest."},
    "P04": {"confidence": 0.88, "reason": "The model answers only from fetched passages, which is grounding in documents."},
    "P05": {"confidence": 0.03, "reason": "The paper is about proteins, unrelated to documents or answering."},
    "P06": {"confidence": 0.04, "reason": "The paper is about image models, not language models answering from documents."},
    "P07": {"confidence": 0.81, "reason": "The paper checks whether answers are supported by the passages they cite."},
    "P08": {"confidence": 0.06, "reason": "The paper is about speech model training, not documents."},
    "P09": {"confidence": 0.4, "reason": "The abstract does not mention documents or retrieval, so it is probably not a match."},
    "P10": {"confidence": 0.08, "reason": "The search here is over network designs, not documents."},
    "P11": {"confidence": 0.58, "reason": "The paper improves dense retrieval, a core part of RAG."},
    "P12": {"confidence": 0.86, "reason": "The paper compares RAG with long-context models for answering from documents."},
    "P13": {"confidence": 0.02, "reason": "The paper is about robotics."},
    "P14": {"confidence": 0.03, "reason": "The paper is about weather forecasting."},
    "P15": {"confidence": 0.35, "reason": "The paper is about hallucinations, but it does not use documents."},
    "P16": {"confidence": 0.79, "reason": "Reranking passages before generation is a part of a RAG system."},
    "P17": {"confidence": 0.45, "reason": "The abstract is about a browsing agent, which is not clearly about documents."},
    "P18": {"confidence": 0.07, "reason": "The paper is about compressing models, not documents."},
    "P19": {"confidence": 0.84, "reason": "The paper evaluates RAG pipelines."},
    "P20": {"confidence": 0.04, "reason": "The paper is about traffic prediction."},
    "P21": {"confidence": 0.77, "reason": "Chunking documents is a part of RAG systems."},
    "P22": {"confidence": 0.03, "reason": "The paper is about music generation."},
    "P23": {"confidence": 0.66, "reason": "The model writes code using retrieved files from the repository, which is answering from given documents."},
    "P24": {"confidence": 0.05, "reason": "The paper is about image generation speed."},
}

by_id = {p["id"]: p for p in papers}
print(f"{len(papers)} labeled papers, {len(v1_verdicts)} v1 verdicts")

Every paper ends up in exactly one of four boxes, depending on the
label and on what v1 did:

|  | v1 suggested it | v1 skipped it |
|---|---|---|
| **Relevant** | correct suggestion | **missed** |
| **Not relevant** | **wrong suggestion** | correct skip |

The two bold boxes are the mistakes. Chapter 5 could only ever show you
wrong suggestions, because they're on the list. Missed papers were
invisible. Let's fill in the boxes:

In [ ]:
def outcomes(threshold=0.5):
    boxes = {"correct suggestion": [], "wrong suggestion": [], "missed": [], "correct skip": []}
    for p in papers:
        suggested = v1_verdicts[p["id"]]["confidence"] >= threshold
        if p["label"] and suggested:
            boxes["correct suggestion"].append(p["id"])
        elif not p["label"] and suggested:
            boxes["wrong suggestion"].append(p["id"])
        elif p["label"] and not suggested:
            boxes["missed"].append(p["id"])
        else:
            boxes["correct skip"].append(p["id"])
    return boxes

boxes = outcomes()
for box, ids in boxes.items():
    print(f"{box:>18}: {len(ids):>2}  {' '.join(ids)}")

print("\nThe mistakes:")
for pid in boxes["wrong suggestion"] + boxes["missed"]:
    v = v1_verdicts[pid]
    print(f"  {pid} ({v['confidence']:.2f}) {by_id[pid]['title']}")
    print(f"       v1 said: {v['reason']}")

There's Chapter 5's answer: on Tuesday, v1 missed P09, and on Wednesday
it missed P17. Both are papers you needed the full text to label in
Chapter 4. The two wrong suggestions are P03 (retrieval practice) and
P11 (product search), the two "retrieval in a different sense" cases.
Four mistakes out of 24, and none of them random.

## From boxes to numbers

Two numbers capture what you care about for a suggestion scout:

- **Precision**: of the papers v1 suggested, how many were actually
  relevant? This is about *trust*. Low precision means you learn to
  ignore its suggestions.
- **Recall**: of the relevant papers, how many did v1 suggest? This is
  about *coverage*. Low recall means good papers slip past you, and you
  won't even know.

You might expect **accuracy**, the share of all verdicts that are
correct, to be the obvious number. Watch what happens to it here:

In [ ]:
def metrics(boxes):
    tp, fp = len(boxes["correct suggestion"]), len(boxes["wrong suggestion"])
    fn, tn = len(boxes["missed"]), len(boxes["correct skip"])
    return {
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "accuracy": (tp + tn) / (tp + fp + fn + tn),
    }

def show(name, m):
    print(f"{name:>18}:  precision {m['precision']:.2f}   recall {m['recall']:.2f}   accuracy {m['accuracy']:.2f}")

show("v1", metrics(outcomes()))
show("skip everything", metrics(outcomes(threshold=2.0)))      # confidence never reaches 2
show("suggest everything", metrics(outcomes(threshold=0.0)))

A scout that does *nothing at all* scores 0.58 accuracy, because most
papers on any day are irrelevant, so "skip" is usually right. Accuracy
rewards that. Precision and recall don't: skipping everything has zero
recall, and suggesting everything has perfect recall but a precision
so low you'd stop reading its list by Wednesday.

v1 lands at 0.80 precision and 0.80 recall: eight of its ten
suggestions were right, and it found eight of the ten relevant papers.
Those two numbers tell you far more than "0.83 accuracy" ever could.

## The threshold knob

v1 suggests a paper when its confidence is at least 0.5. That 0.5 is a
choice, not a law. Lower it and v1 suggests more papers, catching more
relevant ones but also more wrong ones. Raise it and the list gets
shorter and cleaner, but more good papers slip through.

:::{tip} Try it

Change `threshold` and re-run. Then read the full sweep below it. Which
threshold would you pick, and why?
:::

In [ ]:
threshold = 0.5        # try 0.4, 0.6, 0.7

boxes = outcomes(threshold)
show(f"threshold {threshold}", metrics(boxes))
print("    wrong suggestions:", boxes["wrong suggestion"] or "none")
print("    missed:           ", boxes["missed"] or "none")

print("\nFull sweep:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    show(f"threshold {t}", metrics(outcomes(t)))

Precision and recall pull against each other. At 0.7, every suggestion
is right, but v1 finds only seven of the ten relevant papers. At 0.3
it finds all ten, at the price of three wrong suggestions.

And 0.4 looks like a free lunch: better recall *and* better precision
than 0.5. Be suspicious of that. It's better on these 24 papers because
two relevant papers, P09 and P17, happen to sit at 0.40 and 0.45. Pick
a threshold by staring at the exact papers you then measure on, and
you're partly measuring your own tuning. The number will look better
than v1 really is. We'll keep 0.5, chosen before looking.

## Deciding to ship

A score only means something next to a bar you set *before* looking.
For v1 at the "suggest" setting:

- **Recall at least 0.80.** Missing good papers is the costly mistake
  here: you'd never know they existed.
- **Precision at least 0.75.** A wrong suggestion costs you a few
  seconds, so this bar can be a little lower, but not so low that you
  stop trusting the list.

At a higher autonomy setting, like "act", the bars would flip: a wrong
*published* summary is the expensive mistake, so precision would need
to be much higher. The right bar depends on where you stand on the
dial.

In [ ]:
bar_v1 = {"recall": 0.80, "precision": 0.75}

def ready_to_ship(m, bar):
    ok = True
    for name, minimum in bar.items():
        passed = m[name] >= minimum
        ok = ok and passed
        print(f"  {name:>9}: {m[name]:.2f} (needs {minimum:.2f}) -> {'pass' if passed else 'FAIL'}")
    return ok

m = metrics(outcomes(0.5))
print("Ship v1 at 'suggest'?", "YES" if ready_to_ship(m, bar_v1) else "NO")

v1 clears both bars, just barely, so it ships. At this autonomy level,
**shipping** means something modest and concrete: v1 runs every morning
on the new arXiv papers, posts its suggestions to you, and writes
every verdict and trace to its log. It still can't publish or change
anything. You still decide what to read.

That log is the important part. From the next chapter on, the course
switches from our hand-made week to what v1 does in real use.

:::{warning} Where it breaks

The eval says 0.80 precision and 0.80 recall. It's tempting to read
that as "v1 is right 80% of the time". Here's v1 on four papers from
the week after it shipped (same [Colab link](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/06-evals-and-deploy.ipynb) as before, if
you need it):
:::

:::{tip} Predict before you run it

These four papers have no labels: nobody has checked them yet, because
in real use nobody labels papers in advance. What are v1's precision
and recall on them? And do any of them look like a *kind* of paper
that never appeared in our 24?
:::

In [ ]:
next_week = [
    {"id": "N01", "title": "A Survey of Retrieval-Augmented Generation",
     "confidence": 0.93, "reason": "The paper surveys RAG systems."},
    {"id": "N02", "title": "Tuning the Retrieval Stage of Video Recommenders",
     "confidence": 0.71, "reason": "The paper improves a retrieval stage, which matches the interest."},
    {"id": "N03", "title": "Answering Legal Questions from Court Rulings",
     "confidence": 0.48, "reason": "The paper is about law, which seems outside the interest."},
    {"id": "N04", "title": "Sparse Attention for Long Videos",
     "confidence": 0.06, "reason": "The paper is about video models."},
]

for p in next_week:
    verdict = "suggest" if p["confidence"] >= 0.5 else "skip"
    print(f"{p['id']}  {verdict:>7} ({p['confidence']:.2f})  {p['title']}")
    print(f"        {p['reason']}")

You can't compute precision or recall here, because there are no
labels. In real use there never are, unless someone makes them. That's
the first problem: **the eval number describes the 24 papers you
labeled, not the papers v1 sees tomorrow.**

The second problem is quieter. Look at N02 and N03. A recommender
system's "retrieval stage" is another kind of retrieval without a
language model writing anything, the same trap as P11, but in a domain
our dataset never covered. N03 answers legal questions from court
rulings, which is squarely relevant, but v1's reason suggests it
judged the *topic* (law) instead of the *method* (answering from
documents). Neither kind of paper appears in our 24, so the eval had
no way to warn us about either.

Your eval is only as good as the examples in it, and your examples only
contain what you thought of. To know how v1 does on the papers it
actually meets, you have to measure it on real use. That's where
Continuous Calibration begins.

## Quick check

Before moving on, answer for yourself (no peeking ahead):

1. In your own words, what's the difference between precision and recall for the scout? Which mistake does each one count?

<details>
<summary>Check your reasoning ▸</summary>

Precision asks: of what the scout suggested, how much was relevant? It counts wrong suggestions. Recall asks: of what was relevant, how much did the scout suggest? It counts missed papers. A strong answer ties them to the reader's experience: precision is whether you can trust the list, recall is whether good papers slip past you unseen.

</details>

2. A scout that skips every paper gets 0.58 accuracy. Why is accuracy a poor number for this task?

<details>
<summary>Check your reasoning ▸</summary>

Because most papers are irrelevant, so "skip" is usually correct, and accuracy rewards a scout that does nothing. A good answer notes that accuracy hides the two mistakes that actually matter here, missed papers and wrong suggestions, inside one blended number.

</details>

3. Threshold 0.4 beat 0.5 on both precision and recall. Why didn't we switch to it?

<details>
<summary>Check your reasoning ▸</summary>

Because 0.4 was chosen by looking at the same 24 papers it was measured on. It's better mostly because two specific papers sit at 0.40 and 0.45. Tuning to your eval set makes the score look better than the scout will be on new papers. A strong answer adds that the fix is more (or separate) labeled data, not a cleverer threshold.

</details>

4. Why does the recall bar matter more than the precision bar at the "suggest" setting, and how would that change at "act"?

<details>
<summary>Check your reasoning ▸</summary>

At "suggest", a wrong suggestion costs you seconds, but a missed paper is gone for good, so recall is the costlier failure. At "act", a wrong decision becomes a public post, so precision becomes the costly failure and its bar would need to be much higher. The right bar depends on the autonomy level.

</details>

5. v1 passed its bar with 0.80 and 0.80. Why doesn't that mean it will be right 80% of the time next week?

<details>
<summary>Check your reasoning ▸</summary>

The score describes the 24 labeled papers, which only contain the kinds of papers you thought to include. Next week brings papers unlike any of them, like recommender retrieval or legal question answering, and there are no labels to check against. A strong answer says the eval is a starting estimate, and that measuring on real use is the only way to see how v1 actually performs.

</details>

---

**You can now:** evaluate a scout against labeled data with precision and recall, see how a threshold trades one for the other, set a bar that fits the autonomy level, and ship v1 at "suggest".

**But one problem remains:** the eval measured v1 on papers you chose and labeled. Real use brings papers you never thought of, with no labels at all. The next chapter starts Continuous Calibration: scoring v1 on a week of real runs from its log, and seeing how the numbers hold up.

**Next → Chapter 7: Run Evals on Real Use**